In [8]:
import os
import pandas as pd
import json
import numpy as np


#### Baseline

In [9]:
def get_baseline_result_df(dataset,test_file_name='test_result.json'): 
    if dataset == 'brexit':
        result_path = "../results/roberta-base/brexit/Hate/"
        n_anns = 6

    elif dataset == 'mfrc':
        result_path = "../results/roberta-base/mfrc/Moral/"
        n_anns = "24"

        
    
    records = []
    for root, subdirs, files in os.walk(result_path):
        if f'mtl_{n_anns}' in root :
            for file in files:
                    if file == f'{test_file_name}':
                        with open(os.path.join(root, file), 'r') as f:
                            data = json.load(f)
                            seed=root.split('/')[-4]
                            budget = root.split('/')[-3].split('_')[-1]
                            for i,ann in enumerate(data['task_name']):
                                records.append({'seed':seed, 'budget':budget, 'Annotator':ann, 
                                                'f1': data['f1'][i]})
    df = pd.DataFrame(records)
    return df

def get_baseline_performance_variance_across_anns(df):
    df= df.groupby(['seed', 'budget']).agg({'f1': ['mean', 'std']}).reset_index()
    df.columns = df.columns.map('_'.join).str.strip('_')
    df = df.groupby(['budget']).agg({'f1_mean': ['mean', 'std' ]}).reset_index()
    df.columns = df.columns.map('_'.join).str.strip('_')
    df = df.rename(columns={'f1_mean_mean': 'f1_mean', 'f1_mean_std': 'f1_std'})
    return df

### Our method (MTLs)

In [10]:
def get_mtl_result_df(dataset,test_file_name='test_result.json'): 
    if dataset == 'brexit':
        result_path = "../results/roberta-base/brexit/Hate/"
        n_anns = 6

    elif dataset == 'mfrc':
        result_path = "../results/roberta-base/mfrc/Moral/"
        n_anns = 24
        
    records = []
    for root, subdirs, files in os.walk(result_path):
        if (f'mtl' in root) and (f'mtl_{n_anns}' not in root) and ("few_shot") not in root :
            for file in files:
                    if file == f'{test_file_name}':
                        with open(os.path.join(root, file), 'r') as f:
                            data = json.load(f)
                            mtl_tasks = str.join(',',data['task_name'])
                            seed=root.split('/')[-4]
                            budget = root.split('/')[-3].split('_')[-1]
                            for i,ann in enumerate(data['task_name']):
                                records.append({'seed':seed, 'budget':budget, 'Annotator':ann,
                                                'mtl_tasks':mtl_tasks,
                                                'f1': data['f1'][i]})
                                
    df = pd.DataFrame(records)
    return df

### Our method (FS)

In [11]:
def get_fs_result_df(dataset,test_file_name='test_result.json'): 
    if dataset == 'brexit':
        result_path = "../results/roberta-base/brexit/Hate/"
        n_anns = 6

    elif dataset == 'mfrc':
        result_path = "../results/roberta-base/mfrc/Moral/"
        n_anns = 24
        
    records = []
    for root, subdirs, files in os.walk(result_path):
        if "few_shot" in root :
            for file in files:
                    if file == f'{test_file_name}':
                        with open(os.path.join(root, file), 'r') as f:
                            data = json.load(f)
                            annotator = root.split('/')[-2]
                            k_shot = root.split('/')[-3]
                            sampling = root.split('/')[-5]
                            mtl = root.split('/')[-1].split('_')[-2]
                            budget = root.split("/")[-7].split('_')[-1]
                            seed = root.split('/')[-8]
                            
                            records.append({'seed': seed, 'sampling' : sampling, 'budget': budget,
                                            'k_shot': k_shot ,
                                            'Annotator':annotator,  'mtl_tasks': mtl,
                                            'f1': data['f1']})
    df = pd.DataFrame(records)
    return df

In [12]:

def get_mtl_fs_df(df, df_mtl):

    sampling_methods = df['sampling'].unique()
    shots = df['k_shot'].unique()

    dfs_sample_shot = []
    columns = ['seed', 'budget', 'mtl_tasks' ,'Annotator', 'f1']
    for sampling in sampling_methods:
        for shot in shots:
            df_samp_shot = df[(df['sampling'] == sampling) & (df['k_shot'] == shot)]
            df_samp_shot = df_samp_shot[columns]
            df_samp_shot = pd.concat([df_samp_shot, df_mtl])
            
            # first get the average of a single model across annotators
            df_samp_shot = df_samp_shot.groupby(['seed', 'budget', 'mtl_tasks']).agg({'f1': 'mean'}).reset_index()
            
            #get the average the model in one seed
            df_samp_shot = df_samp_shot.groupby(['seed', 'budget']).agg({'f1': ['mean','std']}).reset_index()
            df_samp_shot.columns = df_samp_shot.columns.map('_'.join).str.strip('_')
            
            # get the mean and std across seeds
            df_samp_shot = df_samp_shot.groupby(['budget']).agg({'f1_mean': ['mean', 'std']}).reset_index()
            df_samp_shot.columns = df_samp_shot.columns.map('_'.join).str.strip('_')
            df_samp_shot = df_samp_shot.rename(columns={'f1_mean_mean': 'f1_mean', 'f1_mean_std': 'f1_std'})
            
            df_samp_shot['sampling'] = sampling
            df_samp_shot['k_shot'] = shot
            dfs_sample_shot.append(df_samp_shot)
            
    df_sample_shot = pd.concat(dfs_sample_shot)
    return df_sample_shot



### Brexit

In [13]:
dataset = 'brexit'

df_mtl= get_mtl_result_df(dataset)
df = get_fs_result_df(dataset)

df = get_mtl_fs_df(df, df_mtl)

KeyError: 'sampling'

In [ ]:
df_pivot = df.pivot(index=['sampling', 'k_shot'], columns='budget').sort_index()
df_pivot.columns = df_pivot.columns.map('_'.join).str.strip('_')
df_pivot  = df_pivot.round(3)
df_pivot['0.5']=  df_pivot['f1_mean_0.5'].astype(str) + "_{(" +df_pivot['f1_std_0.5'].astype(str) + ")}"
df_pivot['0.66']=  df_pivot['f1_mean_0.66'].astype(str) + "_{(" +df_pivot['f1_std_0.66'].astype(str) + ")}"
df_pivot['0.83'] = df_pivot['f1_mean_0.83'].astype(str) + "_{(" +df_pivot['f1_std_0.83'].astype(str) + ")}"
df_brexit = df_pivot[['0.5', '0.66', '0.83']]

# df_pivot = df_pivot[['0.5', '0.66', '0.83']].reset_index()
# df_pivot[df_pivot['sampling'] == 'strategy_high_dis']

### MFRC

In [14]:
dataset = 'mfrc'

columns = ['seed', 'budget', 'Annotator', 'f1']

df_mtl= get_mtl_result_df(dataset)
df = get_fs_result_df(dataset)

df = get_mtl_fs_df(df, df_mtl)

KeyError: 'sampling'

In [15]:
df_pivot = df.pivot(index=['sampling', 'k_shot'], columns='budget').sort_index()
df_pivot.columns = df_pivot.columns.map('_'.join).str.strip('_')
df_pivot  = df_pivot.round(3)
df_pivot['0.25']=  df_pivot['f1_mean_0.25'].astype(str) +  "_{(" +df_pivot['f1_std_0.25'].astype(str) + ")}"
df_pivot['0.5']=  df_pivot['f1_mean_0.5'].astype(str) +  "_{(" +df_pivot['f1_std_0.5'].astype(str) + ")}"
df_pivot['0.75'] = df_pivot['f1_mean_0.75'].astype(str) +  "_{(" +df_pivot['f1_std_0.75'].astype(str) + ")}"

df_mfrc = df_pivot[['0.25', '0.5', '0.75']]
# df_pivot = df_pivot[['0.25', '0.5', '0.75']].reset_index()
# df_pivot[df_pivot['sampling'] == 'strategy_balanced']

KeyError: "None of ['sampling', 'k_shot', 'budget'] are in the columns"

In [75]:
df_mfrc
print(pd.merge(df_brexit, df_mfrc, left_index=True, right_index=True).to_latex())

\begin{tabular}{llllllll}
\toprule
 &  & 0.5_x & 0.66 & 0.83 & 0.25 & 0.5_y & 0.75 \\
sampling & k_shot &  &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{strategy_balanced} & 128 & 0.471_{(0.002)} & 0.474_{(0.018)} & 0.468_{(0.014)} & 0.781_{(0.002)} & 0.781_{(0.002)} & 0.782_{(0.003)} \\
 & 16 & 0.443_{(0.005)} & 0.454_{(0.015)} & 0.457_{(0.015)} & 0.777_{(0.002)} & 0.779_{(0.0)} & 0.78_{(0.003)} \\
 & 32 & 0.449_{(0.008)} & 0.458_{(0.009)} & 0.458_{(0.008)} & 0.779_{(0.002)} & 0.78_{(0.001)} & 0.78_{(0.003)} \\
 & 64 & 0.453_{(0.003)} & 0.458_{(0.016)} & 0.459_{(0.011)} & 0.78_{(0.003)} & 0.781_{(0.003)} & 0.781_{(0.004)} \\
\cline{1-8}
\multirow[t]{4}{*}{strategy_high_dis} & 128 & 0.45_{(0.008)} & 0.461_{(0.019)} & 0.466_{(0.016)} & 0.788_{(0.009)} & 0.789_{(0.003)} & 0.783_{(0.005)} \\
 & 16 & 0.421_{(0.01)} & 0.44_{(0.011)} & 0.457_{(0.008)} & 0.784_{(0.01)} & 0.787_{(0.004)} & 0.782_{(0.005)} \\
 & 32 & 0.423_{(0.008)} & 0.44_{(0.015)} & 0.457_{(0.016)} & 0.786_{(0.01)} & 0.788_{(

### Baselines

In [52]:
dataset = 'mfrc'
df = get_baseline_result_df(dataset)
df_mean_var = get_baseline_performance_variance_across_anns(df)
df_mean_var = df_mean_var.round(3)
df_mean_var['f1']  = df_mean_var['f1_mean'].astype(str) + "(" +df_mean_var['f1_std'].astype(str) + ")"
df_mean_var.transpose().drop(['f1_mean', 'f1_std'])

In [ ]:
dataset = 'brexit'
df = get_baseline_result_df(dataset)
df_mean_var = get_baseline_performance_variance_across_anns(df)
df_mean_var = df_mean_var.round(3)
df_mean_var['f1']  = df_mean_var['f1_mean'].astype(str) + "(" +df_mean_var['f1_std'].astype(str) + ")"
df_mean_var.transpose().drop(['f1_mean', 'f1_std'])

### Plot n_mtl vs f1

In [14]:
dataset = 'brexit'

df_mtl= get_mtl_result_df(dataset)
df_mtl['dataset'] = dataset
dataset = 'mfrc'
df_mtl_mfrc = get_mtl_result_df(dataset)
df_mtl_mfrc['dataset'] = dataset

df_concat = pd.concat([df_mtl, df_mtl_mfrc])
df_concat.groupby(['dataset', 'budget']).agg({'f1': ['mean', 'std']}).reset_index()


dataset budget        f1          
                      mean       std
0  brexit    0.5  0.444105  0.135210
1  brexit   0.66  0.452897  0.132360
2  brexit   0.83  0.457452  0.130191
3    mfrc   0.25  0.759980  0.144219
4    mfrc    0.5  0.778484  0.135806
5    mfrc   0.75  0.781630  0.124902

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")
plt.figure(figsize=(6, 5))
colors = {"brexit": "dodgerblue", "mfrc": "orangered"}
ax = sns.lineplot(x="budget", y="f1",
                  hue="dataset", linestyle='--', marker='o', errorbar='sd',
                  data=df_concat, markers=True, dashes=True, palette=colors)
ax.set_ylim(0.2, 0.9)

ax.xaxis.grid(True, linestyle=':', alpha=1)
ax.yaxis.grid(False)
plt.legend(loc='lower left', fontsize=12)
ax.set_xlabel(" % of Annotators")
# plt.xticks(ticks=all_df['budget'], labels=[
#            f'{int(val)}%' for val in all_df['budget']])
ax.tick_params(labelsize=12)
# Display the plot
for i, (x, y) in enumerate(zip(df_concat['budget'], df_concat['f1'])):
    plt.annotate(f'{y}', (x, y), textcoords="offset points", xytext=(0,10), ha='center')

plt.show()